# Bagging, Generalised — and Forests for Regression

## What this notebook adds

The tutorial notebooks built a random forest and explained its two sources of randomness:
bootstrap sampling of rows, and a random subset of features at each split.

Those were presented as **parts of a random forest**. They are actually a general recipe that
works around *any* model, and scikit-learn exposes it separately as `BaggingClassifier` and
`BaggingRegressor`. Seeing the general form makes it much clearer what a forest is: bagging,
with one specific base model and one extra trick.

This notebook also covers the half of the algorithm the lesson has not shown —
**`RandomForestRegressor`** — and the way tuning differs between bagging and boosting.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import (
    BaggingClassifier,
    BaggingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

RSEED = 42

## The data

The loan dataset from the exercise notebook — 9,578 borrowers. We will use it as a
**regression** problem this time: predict the interest rate a borrower was offered.

In [ ]:
loans = pd.read_csv("data/loan_data.csv")

X = loans.drop(columns="int.rate")
y = loans["int.rate"]

preprocessor = make_column_transformer(
    (StandardScaler(), make_column_selector(dtype_include="number")),
    (OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_exclude="number")),
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RSEED
)

print(f"{len(X_train)} training loans, {len(X_test)} held out")
print(f"interest rates run from {y.min():.2%} to {y.max():.2%}")

## Bagging is not about trees

**Bagging** — bootstrap aggregating — is three steps:

1. Draw a bootstrap sample: `n` rows sampled **with replacement**, so roughly a third of the
   original rows are missing and others appear twice.
2. Fit a model on it.
3. Repeat many times, then average the predictions (or vote, for classification).

Nothing in that recipe mentions trees. `BaggingRegressor` takes any estimator you like.

In [ ]:
base_models = {
    "decision tree": DecisionTreeRegressor(random_state=RSEED),
    "linear regression": LinearRegression(),
    "k-nearest neighbours": KNeighborsRegressor(n_neighbors=10),
}

rows = []
for name, base in base_models.items():
    single = make_pipeline(preprocessor, base)
    bagged = make_pipeline(
        preprocessor,
        BaggingRegressor(base, n_estimators=50, random_state=RSEED, n_jobs=-1),
    )
    single.fit(X_train, y_train)
    bagged.fit(X_train, y_train)
    rows.append({
        "base model": name,
        "alone": round(single.score(X_test, y_test), 4),
        "bagged (50)": round(bagged.score(X_test, y_test), 4),
    })

pd.DataFrame(rows)

Read the columns against each other:

| base model | alone | bagged (50) | gain |
|---|---|---|---|
| decision tree | 0.4734 | 0.7357 | **+0.2623** |
| linear regression | 0.6566 | 0.6563 | −0.0003 |
| k-nearest neighbours | 0.6320 | 0.6376 | +0.0056 |

Bagging does **not** help every model equally — it barely helps two of these three.

Bagging reduces **variance** — the part of a model's error that comes from being sensitive to
exactly which rows it was trained on. So:

- An **unpruned decision tree** is almost pure variance: change a few rows and you get a
  different tree. Bagging helps enormously.
- A **linear regression** is stable by construction — the fitted line barely moves when you
  resample. There is almost no variance to remove, so bagging does nearly nothing.

**This is the rule to remember: bag high-variance models.** Bagging a stable model spends
fifty times the compute for no return.

## From bagging to a random forest

A random forest is `BaggingRegressor(DecisionTreeRegressor())` plus **one extra idea**: at
every split, each tree may only consider a random subset of the features.

Why that matters: if one feature is highly predictive, every bagged tree will split on it
first, and the trees end up correlated. Averaging correlated predictions removes less error
than averaging independent ones. Restricting the features at each split **decorrelates** the
trees.

How much that is worth depends on the data, and it is worth checking rather than assuming.

In [ ]:
bagged_trees = make_pipeline(
    preprocessor,
    BaggingRegressor(DecisionTreeRegressor(random_state=RSEED),
                     n_estimators=100, random_state=RSEED, n_jobs=-1),
)
# NOTE: RandomForestRegressor defaults to max_features=1.0 — every feature at every
# split — which is exactly what plain bagged trees do. Set it explicitly, or the
# comparison below is between two identical models.
forest = make_pipeline(
    preprocessor,
    RandomForestRegressor(n_estimators=100, max_features="sqrt",
                          random_state=RSEED, n_jobs=-1),
)

for name, model in [("bagged trees (all features)", bagged_trees),
                    ("random forest (max_features='sqrt')", forest)]:
    model.fit(X_train, y_train)
    print(f"{name:38s} test R² = {model.score(X_test, y_test):.4f}")

## Bagging a whole pipeline

One more thing the general form buys you: the base estimator can be an entire **pipeline**,
not just a bare model. Every bootstrap replicate then re-runs the preprocessing on its own
sample, which is what you want if the preprocessing itself is fitted (an imputer's median, a
scaler's mean).

Note the difference from the cells above, where the preprocessor sat *outside* the bagging
and was therefore fitted once on all the training data.

**One practical catch.** `BaggingRegressor` passes its base estimator a NumPy array rather
than a DataFrame, so anything inside that needs column names — `make_column_selector`, a
`ColumnTransformer` addressed by name — will raise. Either select the columns before bagging,
as below, or address them by integer position.

In [ ]:
# BaggingRegressor hands its base estimator a NumPy array, not a DataFrame — so a
# make_column_selector inside the inner pipeline would fail. Use numeric columns here.
numeric_cols = X.select_dtypes(include="number").columns

inner_pipeline = make_pipeline(
    StandardScaler(),
    DecisionTreeRegressor(max_depth=8, random_state=RSEED),
)

bagged_pipeline = BaggingRegressor(
    inner_pipeline, n_estimators=30, random_state=RSEED, n_jobs=-1
)
bagged_pipeline.fit(X_train[numeric_cols], y_train)

print(f"bagged pipeline test R² = {bagged_pipeline.score(X_test[numeric_cols], y_test):.4f}")
print("each replicate refits the scaler on its own bootstrap sample")

On this dataset the decorrelation buys almost nothing: bagged trees score **0.7409** and the
forest **0.7416**. The `max_features` sweep below is more revealing — `0.5` reaches **0.7519**,
better than both the default and `'sqrt'`.

That is a useful corrective to the usual story. Random feature subsets are a *variance
reduction* device, and their benefit depends on how correlated the trees would otherwise be.
On data with one dominant feature the gain is large; here, where predictive power is spread
across many columns, there is less correlation to remove. **Sweep `max_features` rather than
trusting the default or the folklore.**

## `RandomForestRegressor`

Everything the lesson taught about the classifier carries over. The only changes are in the
leaves and in the aggregation:

| | `RandomForestClassifier` | `RandomForestRegressor` |
|---|---|---|
| each leaf holds | the majority class | the **mean** of its training targets |
| the forest combines by | voting (or averaging probabilities) | **averaging** the trees' predictions |
| split quality measured by | Gini impurity | squared error |
| default `max_features` | `'sqrt'` | `1.0` (all features) |

That last row catches people out — the regressor's default uses **every** feature at each
split, so it is closer to plain bagged trees than the classifier is. If your regression
forest underperforms, setting `max_features` is the first thing to try.

In [ ]:
for max_features in [1.0, 0.5, "sqrt", "log2"]:
    rf = make_pipeline(
        preprocessor,
        RandomForestRegressor(n_estimators=100, max_features=max_features,
                              random_state=RSEED, n_jobs=-1),
    )
    rf.fit(X_train, y_train)
    label = str(max_features)
    print(f"max_features={label:<7} test R² = {rf.score(X_test, y_test):.4f}   "
          f"MAE = {mean_absolute_error(y_test, rf.predict(X_test)):.5f}")

## Tuning bagging is not like tuning boosting

A practical point that matters as soon as you meet gradient boosting in the next lesson.

In a **bagged** ensemble the trees are independent and their predictions are averaged, so
**more trees never makes it worse** — the score rises and then flattens. `n_estimators` is a
compute budget, not a risk. You cannot overfit by adding trees.

In a **boosted** ensemble each tree corrects the previous one's mistakes, so more trees means
a more complex model, and past some point it *will* overfit. There `n_estimators` is a real
hyperparameter that has to be tuned — usually against the learning rate.

Watch the flattening here:

In [ ]:
tree_counts = [1, 5, 10, 25, 50, 100, 200, 400]
scores = []

for n in tree_counts:
    rf = make_pipeline(
        preprocessor,
        RandomForestRegressor(n_estimators=n, random_state=RSEED, n_jobs=-1),
    )
    rf.fit(X_train, y_train)
    scores.append(rf.score(X_test, y_test))
    print(f"n_estimators={n:<5} test R² = {scores[-1]:.4f}")

In [ ]:
plt.plot(tree_counts, scores, marker="o", color="#F5821F", lw=2.5)
plt.xscale("log")
plt.xlabel("n_estimators (log scale)")
plt.ylabel("test R²")
plt.title("More trees never hurts a forest — it just stops helping")
plt.grid(alpha=0.3)
plt.show()

The curve climbs steeply and then flattens. Choose `n_estimators` by asking *how long am I
willing to wait*, and spend your tuning effort on `max_features`, `max_depth` and
`min_samples_leaf` instead — the parameters that actually control the bias-variance
trade-off.

## Summary

- **Bagging** is a general recipe: bootstrap, fit, aggregate. It works around any estimator,
  and `BaggingClassifier` / `BaggingRegressor` expose it directly.
- It reduces **variance**, so it helps unstable models (deep trees) and does almost nothing
  for stable ones (linear regression).
- A **random forest** is bagged trees plus random feature subsets at each split, which
  decorrelates the trees so that averaging removes more error.
- The base estimator can be a **whole pipeline**, in which case each replicate refits the
  preprocessing on its own sample.
- **`RandomForestRegressor`** averages leaf means instead of voting. Its default
  `max_features=1.0` differs from the classifier's `'sqrt'` — worth setting explicitly.
- **More trees never hurts a bagged ensemble.** That will not be true of the boosted models
  in the next lesson.

> **Where this goes next.** The **Ensemble Methods II** lesson builds the other family —
> boosting — where trees are grown in sequence and `n_estimators` becomes a genuine
> hyperparameter. The **Hyperparameter Tuning** lesson is where the search techniques for
> both come from.